In [18]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [34]:
df = pd.read_parquet('../data/all_stocks_data.parquet', engine='fastparquet')

In [35]:
df.shape

(1898322, 7)

In [36]:
df.head()

,Date,Open,High,Low,Close,Volume,Stock
0,2010-01-04,43.193587,43.380729,42.975252,43.157200,3640265.0,MMM
1,2010-01-05,43.042836,43.266370,42.471012,42.886887,3405012.0,MMM
2,2010-01-06,43.604279,43.978564,43.411938,43.495110,6301126.0,MMM
3,2010-01-07,43.313158,43.541891,42.689351,43.526295,5346240.0,MMM
4,2010-01-08,43.505481,43.832981,43.302743,43.832981,4073337.0,MMM


In [37]:
print(df.columns)

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Stock'], dtype='object')


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1898322 entries, 0 to 1898321
Data columns (total 7 columns):
 #   Column  Dtype         
---  ------  -----         
 0   Date    datetime64[ns]
 1   Open    float64       
 2   High    float64       
 3   Low     float64       
 4   Close   float64       
 5   Volume  float64       
 6   Stock   object        
dtypes: datetime64[ns](1), float64(5), object(1)
memory usage: 101.4+ MB


In [39]:
df['Daily_Return'] = df.groupby('Stock')['Close'].pct_change(fill_method=None)

df = df.dropna(subset=['Daily_Return'])

In [40]:
df.head()

,Date,Open,High,Low,Close,Volume,Stock,Daily_Return
1,2010-01-05,43.042836,43.266370,42.471012,42.886887,3405012.0,MMM,-0.006263
2,2010-01-06,43.604279,43.978564,43.411938,43.495110,6301126.0,MMM,0.014182
3,2010-01-07,43.313158,43.541891,42.689351,43.526295,5346240.0,MMM,0.000717
4,2010-01-08,43.505481,43.832981,43.302743,43.832981,4073337.0,MMM,0.007046
5,2010-01-11,43.864168,43.978532,43.359924,43.656231,3500812.0,MMM,-0.004032


In [ ]:
df.isna().sum()

Date            0
Open            0
High            0
Low             0
Close           0
Volume          0
Stock           0
Daily_Return    0
dtype: int64

In [27]:
df = df.dropna(subset=['Daily_Return'])
print(df['Daily_Return'].describe())


count    1.786948e+06
mean     7.205836e-04
std      1.977685e-02
min     -5.386474e-01
25%     -7.999827e-03
50%      7.390572e-04
75%      9.510610e-03
max      7.459325e-01
Name: Daily_Return, dtype: float64


In [33]:
print(df.isna().sum())

Date            0
Open            0
High            0
Low             0
Close           0
Volume          0
Stock           0
Daily_Return    0
is_missing      0
dtype: int64


In [32]:
# Flag missing Close values
df['is_missing'] = df['Close'].isna().astype(int)

# Identify stretches (runs) of missing data within each ticker
def find_missing_stretches(group):
    # Identify where a new stretch starts (when previous row was not missing)
    group['gap_id'] = (group['is_missing'].ne(group['is_missing'].shift())).cumsum()
    
    # Filter only missing rows
    missing_stretches = (
        group[group['is_missing'] == 1]
        .groupby('gap_id')
        .agg(
            Stock=('Stock', 'first'),
            start_date=('Date', 'min'),
            end_date=('Date', 'max'),
            missing_days=('Date', 'count')
        )
        .reset_index(drop=True)
    )
    return missing_stretches

# Apply per Ticker
missing_summary = df.groupby('Stock', group_keys=False).apply(find_missing_stretches)

# Sort by longest gaps
missing_summary = missing_summary.sort_values('missing_days', ascending=False)

# Show summary statistics
print("Top 10 longest missing data stretches:")
print(missing_summary)

print("\nSummary of missing streaks by length:")
print(missing_summary['missing_days'].describe())

# (Optional) Count how many tickers have long missing periods
long_gaps = missing_summary[missing_summary['missing_days'] > 10]
tickers_with_long_gaps = long_gaps['Stock'].nunique()
print(f"\nNumber of tickers with gaps > 10 days: {tickers_with_long_gaps}")

Top 10 longest missing data stretches:
Empty DataFrame
Columns: [Stock, start_date, end_date, missing_days]
Index: []

Summary of missing streaks by length:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: missing_days, dtype: float64

Number of tickers with gaps > 10 days: 0
